In [ ]:
import sqlite3
import pandas as pd
import numpy as np

database_path = "ShiftData.db"
connection = sqlite3.connect(database_path)

shift_data = pd.read_sql_query(
    "SELECT * FROM ShiftPerformance",
    connection
)
connection.close()
shift_data.head()


In [ ]:
shift_data.info()


In [ ]:
print("Number of rows", shift_data.shape[0])
print("Number of columns:", shift_data.shape[1] )



In [ ]:
column_summary= pd.DataFrame({
"Data_Type": shift_data.dtypes,
"Non_Missing_Values": shift_data.notna().sum(),
"Missin_Values": shift_data.isna().sum(),
"Missing_Percentage":(shift_data.isna().mean() * 100).round(2),
"Unique_Values": shift_data.nunique()
})
column_summary


In [ ]:
categorical_columns = shift_data.select_dtypes(include= ["object"]).columns

categorical_columns.tolist()


In [ ]:

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(
        shift_data[column]
        .value_counts(dropna=False)
        .head(20)
    )


In [ ]:
numerical_columns = shift_data.select_dtypes(
    include=["number"]
).columns

numerical_summary = shift_data[
    numerical_columns
].describe().T

numerical_summary


In [ ]:
print(
    "Completely duplicated rows:",
    shift_data.duplicated().sum()
)

print(
    "Unique production IDs:",
    shift_data["production_id"].nunique()
)

print(
    "Repeated production IDs:",
    shift_data["production_id"].duplicated().sum()
)

print(
    "Unique quality-control IDs:",
    shift_data["qc_id"].nunique()
)


In [ ]:
quality_checks = {
    "Negative units produced":
        (shift_data["units_produced"] < 0).sum(),

    "Negative defect counts":
        (shift_data["defect_count"] < 0).sum(),

    "Defects greater than units produced":
        (
            shift_data["defect_count"]
            > shift_data["units_produced"]
        ).sum(),

    "Runtime below 0 hours":
        (shift_data["runtime_hours"] < 0).sum(),

    "Runtime above 8 hours":
        (shift_data["runtime_hours"] > 8).sum(),

    "Downtime below 0 minutes":
        (shift_data["downtime_minutes"] < 0).sum(),

    "Downtime above 480 minutes":
        (shift_data["downtime_minutes"] > 480).sum(),

    "Humidity below 0%":
        (shift_data["humidity"] < 0).sum(),

    "Humidity above 100%":
        (shift_data["humidity"] > 100).sum(),

    "Efficiency score below 0":
        (shift_data["shift_efficiency_score"] < 0).sum(),

    "Efficiency score above 100":
        (shift_data["shift_efficiency_score"] > 100).sum()
}

pd.Series(quality_checks)


In [ ]:
efficiency_above_100 = shift_data[
    shift_data["shift_efficiency_score"] > 100
]

efficiency_above_100[
    [
        "shift_efficiency_score",
        "units_produced",
        "defect_count",
        "runtime_hours",
        "downtime_minutes"
    ]
].head(10)


In [ ]:
print("Dataset shape:", shift_data.shape)
print("Exact duplicate rows:", shift_data.duplicated().sum())
print("\nColumn names and data types:")
shift_data.info()


In [ ]:
shift_data["start_datetime"] = pd.to_datetime(
    shift_data["date"].astype(str) + " " + shift_data["start_time"].astype(str),
    errors="coerce"
)

shift_data["end_datetime"] = pd.to_datetime(
    shift_data["date"].astype(str) + " " + shift_data["end_time"].astype(str),
    errors="coerce"
)


In [ ]:
overnight_shifts = shift_data["end_datetime"] <= shift_data["start_datetime"]

shift_data.loc[overnight_shifts, "end_datetime"] += pd.Timedelta(days=1)


In [ ]:
shift_data[
    ["date", "start_time", "end_time", "start_datetime", "end_datetime"]
].head()


In [ ]:
shift_data["timestamp"] = pd.to_datetime(
    shift_data["timestamp"],
    errors="coerce"
)


In [ ]:
shift_data[
    ["start_datetime", "end_datetime", "timestamp"]
].dtypes


In [ ]:
shift_data_clean = shift_data.copy()


In [ ]:
missing_report = pd.DataFrame({
    "missing_count": shift_data_clean.isnull().sum(),
    "missing_percentage": (
        shift_data_clean.isnull().mean() * 100
    ).round(2)
})

missing_report = missing_report[
    missing_report["missing_count"] > 0
]

missing_report


In [ ]:
pd.crosstab(
    shift_data_clean["maintenance_flag"],
    shift_data_clean["maintenance_id"].isna()
)


In [ ]:
no_maintenance = shift_data_clean["maintenance_flag"] == 0

shift_data_clean.loc[
    no_maintenance, "issue_type"
] = "No maintenance"

shift_data_clean.loc[
    no_maintenance, "resolved_by"
] = "Not applicable"

shift_data_clean.loc[
    no_maintenance, "maintenance_downtime"
] = 0


In [ ]:
shift_data_clean["maintenance_id"] = (
    shift_data_clean["maintenance_id"].astype("Int64")
)


In [ ]:
shift_data_clean["temperature"] = (
    shift_data_clean["temperature"].fillna(
        shift_data_clean["temperature"].median()
    )
)

shift_data_clean["humidity"] = (
    shift_data_clean["humidity"].fillna(
        shift_data_clean["humidity"].median()
    )
)


In [ ]:
shift_data_clean["timestamp"] = pd.to_datetime(
    shift_data_clean["timestamp"],
    errors="coerce"
)


In [ ]:
pd.DataFrame({
    "missing_count": shift_data_clean.isnull().sum(),
    "missing_percentage": (
        shift_data_clean.isnull().mean() * 100
    ).round(2)
}).query("missing_count > 0")


In [ ]:
shift_data_clean = shift_data.copy()


In [ ]:
no_maintenance = (
    (shift_data_clean["maintenance_flag"] == 0) &
    (shift_data_clean["maintenance_id"].isna())
)


In [ ]:
shift_data_clean.loc[
    no_maintenance, "issue_type"
] = "No maintenance"

shift_data_clean.loc[
    no_maintenance, "resolved_by"
] = "Not applicable"

shift_data_clean.loc[
    no_maintenance, "maintenance_downtime"
] = 0

shift_data_clean["maintenance_id"] = (
    shift_data_clean["maintenance_id"].astype("Int64")
)


In [ ]:
shift_data_clean["temperature"] = (
    shift_data_clean["temperature"].fillna(
        shift_data_clean["temperature"].median()
    )
)

shift_data_clean["humidity"] = (
    shift_data_clean["humidity"].fillna(
        shift_data_clean["humidity"].median()
    )
)

shift_data_clean["timestamp"] = pd.to_datetime(
    shift_data_clean["timestamp"],
    errors="coerce"
)


In [ ]:
print("Dataset shape:", shift_data_clean.shape)
print("Exact duplicate rows:", shift_data_clean.duplicated().sum())

print("\nRemaining missing values:")
display(
    pd.DataFrame({
        "missing_count": shift_data_clean.isnull().sumelum(),
        " multiline";
    })
)


In [ ]:
print("Dataset shape:", shift_data_clean.shape)
print("Exact duplicate rows:", shift_data_clean.duplicated().sum())

print("\nRemaining missing values:")

final_missing_report = pd.DataFrame({
    "missing_count": shift_data_clean.isnull().sum(),
    "missing_percentage": (
        shift_data_clean.isnull().mean() * 100
    ).round(2)
})

display(
    final_missing_report[
        final_missing_report["missing_count"] > 0
    ]
)


In [ ]:
overview = pd.DataFrame({
    "Measure": [
        "Rows",
        "Source columns",
        "Exact duplicate rows"
    ],
    "Result": [
        shift_data.shape[0],
        shift_data.shape[1],
        shift_data.duplicated().sum()
    ]
})

overview


In [ ]:
structure = pd.DataFrame({
    "data_type": shift_data.dtypes.astype(str),
    "non_null_count": shift_data.notna().sum(),
    "missing_count": shift_data.isna().sum()
})

structure


In [ ]:
target_summary = (
    shift_data["shift_efficiency_score"]
    .describe()
    .round(2)
    .to_frame("value")
)

target_summary.loc["above_100", "value"] = (
    shift_data["shift_efficiency_score"] > 100
).sum()

target_summary


In [ ]:
ax = shift_data["shift_efficiency_score"].plot(
    kind="hist",
    bins=30,
    color="#4C78A8",
    edgecolor="white",
    figsize=(8, 4)
)

ax.axvline(
    100,
    color="#C44E52",
    linestyle="--",
    label="Score = 100"
)

ax.set(
    title="Distribution of Shift Efficiency Score",
    xlabel="Shift efficiency score"
)

ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
categorical_columns = [
    "shift_name",
    "skill_category",
    "machine_status",
    "issue_type",
    "defect_type",
    "severity",
    "inspection_result"
]

categorical_summary = pd.DataFrame({
    "unique_non_null":
        shift_data[categorical_columns].nunique(),

    "most_common": [
        shift_data[column].mode(dropna=True).iloc[0]
        for column in categorical_columns
    ],

    "most_common_count": [
        shift_data[column]
        .value_counts(dropna=True)
        .iloc[0]
        for column in categorical_columns
    ]
})Z

categorical_summary


In [ ]:
identifier_columns = [
    "shift_id",
    "supervisor_id",
    "production_id",
    "operator_id",
    "machine_id",
    "maintenance_id",
    "qc_id"
]

identifier_summary = pd.DataFrame({
    "unique_non_null":
        shift_data[identifier_columns].nunique(),

    "missing_count":
        shift_data[identifier_columns].isna().sum()
})

identifier_summary


In [ ]:
shift_data["start_datetime"] = pd.to_datetime(
    shift_data["date"].astype(str)
    + " "
    + shift_data["start_time"].astype(str),
    errors="coerce"
)

shift_data["end_datetime"] = pd.to_datetime(
    shift_data["date"].astype(str)
    + " "
    + shift_data["end_time"].astype(str),
    errors="coerce"
)

overnight_shifts = (
    shift_data["end_datetime"]
    <= shift_data["start_datetime"]
)

shift_data.loc[
    overnight_shifts,
    "end_datetime"
] += pd.Timedelta(days=1)

shift_data["timestamp"] = pd.to_datetime(
    shift_data["timestamp"],
    errors="coerce"
)


In [ ]:
time_validation = pd.Series({
    "Earliest shift start":
        shift_data["start_datetime"].min(),

    "Latest shift end":
        shift_data["end_datetime"].max(),

    "Overnight rows adjusted":
        int(overnight_shifts.sum()),

    "Invalid start datetimes":
        int(shift_data["start_datetime"].isna().sum()),

    "Invalid end datetimes":
        int(shift_data["end_datetime"].isna().sum()),

    "Missing environmental timestamps":
        int(shift_data["timestamp"].isna().sum())
})

time_validation


In [ ]:
relationship_features = [
    "units_produced",
    "defect_count",
    "cycle_time_avg",
    "experience_level",
    "runtime_hours",
    "downtime_minutes",
    "maintenance_flag",
    "maintenance_downtime",
    "temperature",
    "humidity"
]

target_correlations = (
    shift_data[
        relationship_features
        + ["shift_efficiency_score"]
    ]
    .corr(numeric_only=True)["shift_efficiency_score"]
    .drop("shift_efficiency_score")
    .sort_values(ascending=False)
    .round(3)
)

target_correlations


In [ ]:
quality_checks = pd.Series({
    "Negative units produced":
        (shift_data["units_produced"] < 0).sum(),

    "Negative defect counts":
        (shift_data["defect_count"] < 0).sum(),

    "Defects greater than units produced":
        (
            shift_data["defect_count"]
            > shift_data["units_produced"]
        ).sum(),

    "Runtime below 0 hours":
        (shift_data["runtime_hours"] < 0).sum(),

    "Runtime above 8 hours":
        (shift_data["runtime_hours"] > 8).sum(),

    "Downtime below 0 minutes":
        (shift_data["downtime_minutes"] < 0).sum(),

    "Downtime above 480 minutes":
        (shift_data["downtime_minutes"] > 480).sum(),

    "Humidity below 0%":
        (shift_data["humidity"] < 0).sum(),

    "Humidity above 100%":
        (shift_data["humidity"] > 100).sum(),

    "Efficiency score below 0":
        (
            shift_data["shift_efficiency_score"] < 0
        ).sum(),

    "Efficiency score above 100":
        (
            shift_data["shift_efficiency_score"] > 100
        ).sum()
})

quality_checks


In [ ]:
missing_report = pd.DataFrame({
    "missing_count":
        shift_data.isna().sum(),

    "missing_percentage":
        (
            shift_data.isna().mean() * 100
        ).round(2)
})

missing_report = missing_report.query(
    "missing_count > 0"
).sort_values(
    "missing_percentage",
    ascending=False
)

missing_report


In [ ]:
pd.crosstab(
    shift_data["maintenance_flag"],
    shift_data["maintenance_id"].isna()
)


In [ ]:
shift_data_clean = shift_data.copy()

no_maintenance = (
    (shift_data_clean["maintenance_flag"] == 0)
    &
    (shift_data_clean["maintenance_id"].isna())
)

shift_data_clean.loc[
    no_maintenance,
    "issue_type"
] = "No maintenance"

shift_data_clean.loc[
    no_maintenance,
    "resolved_by"
] = "Not applicable"

shift_data_clean.loc[
    no_maintenance,
    "maintenance_downtime"
] = 0

shift_data_clean["maintenance_id"] = (
    shift_data_clean["maintenance_id"]
    .astype("Int64")
)

for column in ["temperature", "humidity"]:
    shift_data_clean[column] = (
        shift_data_clean[column].fillna(
            shift_data_clean[column].median()
        )
    )


In [ ]:
final_missing_report = pd.DataFrame({
    "missing_count":
        shift_data_clean.isna().sum(),

    "missing_percentage":
        (
            shift_data_clean.isna().mean() * 100
        ).round(2)
}).query("missing_count > 0")

print(
    "Working rows:",
    shift_data_clean.shape[0]
)

print(
    "Working columns:",
    shift_data_clean.shape[1]
)

print(
    "Exact duplicate rows:",
    shift_data_clean.duplicated().sum()
)

display(final_missing_report)


In [ ]:
business_numeric = [
    "units_produced",
    "defect_count",
    "cycle_time_avg",
    "experience_level",
    "runtime_hours",
    "downtime_minutes",
    "maintenance_downtime",
    "temperature",
    "humidity"
]

numeric_summary = (
    shift_data[business_numeric]
    .describe()
    .T[
        ["count", "mean", "std", "min", "50%", "max"]
    ]
    .round(2)
)

numeric_summary


In [ ]:
identifier_columns = [
    "shift_id",
    "supervisor_id",
    "production_id",
    "operator_id",
    "machine_id",
    "maintenance_id",
    "qc_id"
]

identifier_summary = pd.DataFrame({
    "unique_non_null":
        shift_data[identifier_columns].nunique(),

    "missing_count":
        shift_data[identifier_columns].isna().sum()
})

identifier_summary


In [ ]:
shift_data_clean = shift_data.copy()

no_maintenance = (
    (shift_data_clean["maintenance_flag"] == 0)
    &
    (shift_data_clean["maintenance_id"].isna())
)

shift_data_clean.loc[
    no_maintenance,
    "issue_type"
] = "No maintenance"

shift_data_clean.loc[
    no_maintenance,
    "resolved_by"
] = "Not applicable"

shift_data_clean.loc[
    no_maintenance,
    "maintenance_downtime"
] = 0

shift_data_clean["maintenance_id"] = (
    shift_data_clean["maintenance_id"]
    .astype("Int64")
)

for column in ["temperature", "humidity"]:
    shift_data_clean[column] = (
        shift_data_clean[column].fillna(
            shift_data_clean[column].median()
        )
    )


In [ ]:
final_missing_report = pd.DataFrame({
    "missing_count":
        shift_data_clean.isna().sum(),

    "missing_percentage":
        (
            shift_data_clean.isna().mean() * 100
        ).round(2)
}).query("missing_count > 0")

print(
    "Working rows:",
    shift_data_clean.shape[0]
)

print(
    "Working columns:",
    shift_data_clean.shape[1]
)

print(
    "Exact duplicate rows:",
    shift_data_clean.duplicated().sum()
)

display(final_missing_report)

